# Designing and testing your own vaccines

The [vaccines guide](12_vaccines.ipynb) shows every kind of detector. This one is about the craft: turning a rule your
app needs into a vaccine you can trust, step by step, with one realistic example. Everything here ships in
`pip install immune-ai`. Your vaccines live in your own repository, and you don't contribute anything to Immune.

| Step | The question to answer | Where it goes |
| --- | --- | --- |
| [1](#1.-Name-the-rule) | What exactly must never happen? | `title` and `description` |
| [2](#2.-Decide-where-to-look) | Where in the call does it show up? | `stage` |
| [3](#3.-Choose-how-to-recognize-it) | What recognizes it: words, a format, meaning, a tool call? | `detect` |
| [4](#4.-Write-examples-that-should-and-shouldn't-fire) | What must fire, and what must pass? | `tests` |
| [5](#5.-Test-the-examples) | Do the examples behave? | `immune vaccines test` |
| [6](#6.-Trial-it-on-everyday-traffic) | Does it leave normal traffic alone? | `immune vaccines trial` |
| [7](#7.-Test-it-inside-your-app's-test-suite) | Does your app react the way you want? | your tests |
| [8](#8.-Ship-it-observed,-then-enforce-it) | Are real hits right? Then act on them | `enforce` |

Cells marked **live** ask Jev and need `TYPESAFE_API_KEY` (a few calls, a fraction of a cent). Everything else runs
offline. No cell calls OpenAI: the app's replies come from Immune's test kit.

## 1. Name the rule

Pinecrest Outfitters' support assistant answers questions about orders. Customers love a confident answer, so it
sometimes says "your tent arrives Friday". Pinecrest can't promise that, because the courier decides. That's the rule.

Write it as one sentence about one behavior you can observe:

| Too vague | Testable |
| --- | --- |
| "Be careful about shipping" | "The reply promises the customer a delivery date or day" |
| "Don't overpromise" | "The reply promises a refund the policy doesn't allow" |
| "Keep it professional" | "The reply uses profanity" |

A good rule also has an obvious **near miss**: something close that must pass. Here, quoting the usual delivery window
("3 to 5 working days") is fine. If you can't name a near miss, the rule is probably too broad.

## 2. Decide where to look

Every vaccine watches one place in the call:

| `stage` | Watches | Pinecrest example |
| --- | --- | --- |
| `input` | The user's message | A customer pastes a gift-card code into the chat |
| `data` | Documents and tool results the model reads | A product review that tells the assistant what to say |
| `tool` | Tool calls the model makes | A refund over $200 without a person |
| `output` | The reply | **A promised delivery date** |

The delivery promise is something the assistant *says*, so it's an `output` vaccine.

## 3. Choose how to recognize it

Use the simplest detector that works:

| If you can… | Use | Runs | Cost |
| --- | --- | --- | --- |
| List the words or names | `keywords` | In your process | None |
| Describe a text format | `regex` | In your process | None |
| Spot words that are sometimes innocent | `keywords` + `confirm: jev` | Words find candidates, Jev confirms | One question, only on a match |
| Only judge it by meaning | `questions` | Jev, in its existing request for the stage | One question per call |
| Check a tool call's arguments | `tool` + `argument` | In your process | None |
| Express it in code | `python` | In your process | Your function's time |

A delivery promise can be worded a hundred ways ("by Friday", "tomorrow", "no later than the 14th"), so a word list
won't do. It needs **a question for Jev**. Write the question as a yes/no statement about one thing:

| Weak question | Why | Better |
| --- | --- | --- |
| "The reply mentions delivery." | True of every delivery answer, including the fine ones | "The assistant output promises the customer that their order will arrive by a specific date or day." |
| "The reply promises a date or a price." | Two behaviors in one question | One vaccine per behavior |
| "Is the reply bad?" | Jev can't know your policy | Describe the behavior itself |

### Compare: a rule a computer can check

Not every rule needs Jev. "Refunds over $200 need a person" is a tool rule, checked in your process, with no key and
no cost. Its examples are tool calls, and they're tested for real, offline:

In [1]:
import tempfile
from pathlib import Path

folder = Path(tempfile.mkdtemp())
(folder / "vaccines").mkdir()
(folder / "vaccines" / "large_refunds.yaml").write_text("""
id: pinecrest.large_refunds
title: A refund over $200 without a person
stage: tool
detect:
  tool: refund_order
  argument: {path: amount, greater_than: 200}
respond:
  tool: confirm
  message: "Refunds over $200 need a team member to approve them."
tests:
  positives:
    - {tool: refund_order, arguments: {order_id: "5521", amount: 349}}
  negatives:
    - {tool: refund_order, arguments: {order_id: "5521", amount: 49}}
    - {tool: refund_order, arguments: {order_id: "5521", amount: 200}}
""")
!cd {folder} && immune vaccines test vaccines/

vaccine                  expect  example                                           result  detail
───────────────────────  ──────  ────────────────────────────────────────────────  ──────  ─────────────────────────────────────
pinecrest.large_refunds  fire    refund_order {"amount": 349, "order_id": "5521"}  pass    fired p=1.00 refund_order: amount=349
pinecrest.large_refunds  quiet   refund_order {"amount": 49, "order_id": "5521"}   pass    did not fire
pinecrest.large_refunds  quiet   refund_order {"amount": 200, "order_id": "5521"}  pass    did not fire

1 vaccines, 3 examples: all passed


**What you see:** every example ran through a real Immune runtime, offline. The $200 refund passes because the rule
says *over* $200: testing the boundary is a cheap way to catch an off-by-one.

## 4. Write examples that should and shouldn't fire

Examples are the vaccine's specification. Write two kinds:

- **Positives** must fire. Vary the wording: a date, a weekday, "tomorrow".
- **Negatives** must pass. Make them *near misses*, not random text: the delivery window, a shipping confirmation, a
  pointer to tracking. They're what make a vaccine safe to switch on.

Here is the complete delivery vaccine:

In [2]:
import tempfile
from pathlib import Path

folder = Path(tempfile.mkdtemp())
(folder / "vaccines").mkdir()
(folder / "vaccines" / "no_delivery_promises.yaml").write_text("""
id: pinecrest.no_delivery_promises
title: The reply promises a delivery date
description: >-
  Couriers set delivery dates, so the assistant must not promise one. It may quote the
  usual delivery window, share tracking, or say when an order shipped.
stage: output
detect:
  questions:
    - key: promises_date
      text: "The assistant output promises the customer that their order will arrive by a specific date or day."
respond:
  message: "Orders usually arrive in 3 to 5 working days; your tracking link is in your email."
tests:
  positives:
    - "Your tent will arrive on Friday, guaranteed."
    - "It'll be with you by 14 March at the latest."
    - "Don't worry, your order arrives tomorrow."
  negatives:
    - "Orders usually arrive in 3 to 5 working days."
    - "Your order shipped on Monday; the tracking link is in your email."
    - "The courier's website shows the latest delivery estimate."
""")
print((folder / "vaccines" / "no_delivery_promises.yaml").read_text())


id: pinecrest.no_delivery_promises
title: The reply promises a delivery date
description: >-
  Couriers set delivery dates, so the assistant must not promise one. It may quote the
  usual delivery window, share tracking, or say when an order shipped.
stage: output
detect:
  questions:
    - key: promises_date
      text: "The assistant output promises the customer that their order will arrive by a specific date or day."
respond:
  message: "Orders usually arrive in 3 to 5 working days; your tracking link is in your email."
tests:
  positives:
    - "Your tent will arrive on Friday, guaranteed."
    - "It'll be with you by 14 March at the latest."
    - "Don't worry, your order arrives tomorrow."
  negatives:
    - "Orders usually arrive in 3 to 5 working days."
    - "Your order shipped on Monday; the tracking link is in your email."
    - "The courier's website shows the latest delivery estimate."



## 5. Test the examples

`immune vaccines test` runs each example through a private Immune runtime. Offline, a scripted stand-in answers Jev's
question. That checks the file and the wiring (the stage, the head, the response), not whether the question captures
your rule:

In [3]:
import tempfile
from pathlib import Path

folder = Path(tempfile.mkdtemp())
(folder / "vaccines").mkdir()
(folder / "vaccines" / "no_delivery_promises.yaml").write_text("""
id: pinecrest.no_delivery_promises
title: The reply promises a delivery date
description: >-
  Couriers set delivery dates, so the assistant must not promise one. It may quote the
  usual delivery window, share tracking, or say when an order shipped.
stage: output
detect:
  questions:
    - key: promises_date
      text: "The assistant output promises the customer that their order will arrive by a specific date or day."
respond:
  message: "Orders usually arrive in 3 to 5 working days; your tracking link is in your email."
tests:
  positives:
    - "Your tent will arrive on Friday, guaranteed."
    - "It'll be with you by 14 March at the latest."
    - "Don't worry, your order arrives tomorrow."
  negatives:
    - "Orders usually arrive in 3 to 5 working days."
    - "Your order shipped on Monday; the tracking link is in your email."
    - "The courier's website shows the latest delivery estimate."
""")
!cd {folder} && immune vaccines test vaccines/

vaccine                         expect  example                                                       result  detail
──────────────────────────────  ──────  ────────────────────────────────────────────────────────────  ──────  ───────────────────────────────────────────────────────────────
pinecrest.no_delivery_promises  fire    Your tent will arrive on Friday, guaranteed.                  pass    fired p=0.97 pinecrest_no_delivery_promises__promises_date=0.97
pinecrest.no_delivery_promises  fire    It'll be with you by 14 March at the latest.                  pass    fired p=0.97 pinecrest_no_delivery_promises__promises_date=0.97
pinecrest.no_delivery_promises  fire    Don't worry, your order arrives tomorrow.                     pass    fired p=0.97 pinecrest_no_delivery_promises__promises_date=0.97
pinecrest.no_delivery_promises  quiet   Orders usually arrive in 3 to 5 working days.                 pass    did not fire
pinecrest.no_delivery_promises  quiet   Your order shipped on Mo

Now the real check. **Live**: with `--live`, Jev judges every example, so you learn whether the question means what you
meant. If a positive is missed or a near miss fires, sharpen the question (or the example) and run it again:

In [4]:
import tempfile
from pathlib import Path

folder = Path(tempfile.mkdtemp())
(folder / "vaccines").mkdir()
(folder / "vaccines" / "no_delivery_promises.yaml").write_text("""
id: pinecrest.no_delivery_promises
title: The reply promises a delivery date
description: >-
  Couriers set delivery dates, so the assistant must not promise one. It may quote the
  usual delivery window, share tracking, or say when an order shipped.
stage: output
detect:
  questions:
    - key: promises_date
      text: "The assistant output promises the customer that their order will arrive by a specific date or day."
respond:
  message: "Orders usually arrive in 3 to 5 working days; your tracking link is in your email."
tests:
  positives:
    - "Your tent will arrive on Friday, guaranteed."
    - "It'll be with you by 14 March at the latest."
    - "Don't worry, your order arrives tomorrow."
  negatives:
    - "Orders usually arrive in 3 to 5 working days."
    - "Your order shipped on Monday; the tracking link is in your email."
    - "The courier's website shows the latest delivery estimate."
""")
!cd {folder} && immune vaccines test vaccines/ --live

vaccine                         expect  example                                                       result  detail
──────────────────────────────  ──────  ────────────────────────────────────────────────────────────  ──────  ───────────────────────────────────────────────────────────────
pinecrest.no_delivery_promises  fire    Your tent will arrive on Friday, guaranteed.                  pass    fired p=0.94 pinecrest_no_delivery_promises__promises_date=0.94
pinecrest.no_delivery_promises  fire    It'll be with you by 14 March at the latest.                  pass    fired p=0.85 pinecrest_no_delivery_promises__promises_date=0.85
pinecrest.no_delivery_promises  fire    Don't worry, your order arrives tomorrow.                     pass    fired p=0.92 pinecrest_no_delivery_promises__promises_date=0.92
pinecrest.no_delivery_promises  quiet   Orders usually arrive in 3 to 5 working days.                 pass    did not fire
pinecrest.no_delivery_promises  quiet   Your order shipped on Mo

## 6. Trial it on everyday traffic

Tests prove it catches what you meant. A trial shows it leaves everything else alone: it runs the vaccine over Immune's
packaged everyday samples plus your own traffic, and reports how often it would fire. Use masked samples of your real
replies, one per line or as JSON Lines. `--max-rate` makes the command fail above a rate, which is handy in CI.
**Live:**

In [5]:
import json
import tempfile
from pathlib import Path

folder = Path(tempfile.mkdtemp())
(folder / "vaccines").mkdir()
(folder / "vaccines" / "no_delivery_promises.yaml").write_text("""
id: pinecrest.no_delivery_promises
title: The reply promises a delivery date
description: >-
  Couriers set delivery dates, so the assistant must not promise one. It may quote the
  usual delivery window, share tracking, or say when an order shipped.
stage: output
detect:
  questions:
    - key: promises_date
      text: "The assistant output promises the customer that their order will arrive by a specific date or day."
respond:
  message: "Orders usually arrive in 3 to 5 working days; your tracking link is in your email."
tests:
  positives:
    - "Your tent will arrive on Friday, guaranteed."
    - "It'll be with you by 14 March at the latest."
    - "Don't worry, your order arrives tomorrow."
  negatives:
    - "Orders usually arrive in 3 to 5 working days."
    - "Your order shipped on Monday; the tracking link is in your email."
    - "The courier's website shows the latest delivery estimate."
""")
everyday = [
    "Your order shipped this morning.",
    "Standard delivery takes 3 to 5 working days.",
    "Express delivery costs $9.95 and usually takes 1 to 2 working days.",
    "The Ridgeline 2P tent weighs 1.9 kg and packs down to 45 cm.",
    "You can return unused items within 60 days for a full refund.",
    "Our High Street store opens at 9am.",
    "The courier will email you a delivery window on the day.",
    "Click and collect orders are usually ready within two hours.",
    "Gift cards never expire and work online and in store.",
    "I've updated the delivery address on order 5521.",
    "Tracking updates can take up to 24 hours to appear.",
    "We deliver to the UK and Ireland.",
]
(folder / "traffic.jsonl").write_text("".join(json.dumps({"text": line}) + "\n" for line in everyday))
!cd {folder} && immune vaccines trial vaccines/no_delivery_promises.yaml --corpus traffic.jsonl --max-rate 0.02 --live

trial pinecrest.no_delivery_promises (live): 40 everyday output samples, 12 from traffic.jsonl
fired on 0 of 52 samples (0.0%)
Jev tokens: 36,979


**What you see:** how many samples fired, and which. A vaccine that fires on ordinary replies isn't ready: narrow the
question, or add the reply that fired as a near miss and test again.

## 7. Test it inside your app's test suite

Your own tests can check what your app *does* when the vaccine fires. Immune's test kit replaces the model with a
scripted reply and Jev with a scripted answer, so the test is fast, offline and repeatable. The answer is keyed by the
question's `key` (`promises_date`):

In [6]:
import tempfile
from pathlib import Path

import immune
from immune.testing import FakeReply, ImmuneHarness, MockSensor

folder = Path(tempfile.mkdtemp())
(folder / "vaccines").mkdir()
(folder / "vaccines" / "no_delivery_promises.yaml").write_text("""
id: pinecrest.no_delivery_promises
title: The reply promises a delivery date
description: >-
  Couriers set delivery dates, so the assistant must not promise one. It may quote the
  usual delivery window, share tracking, or say when an order shipped.
stage: output
detect:
  questions:
    - key: promises_date
      text: "The assistant output promises the customer that their order will arrive by a specific date or day."
respond:
  message: "Orders usually arrive in 3 to 5 working days; your tracking link is in your email."
tests:
  positives:
    - "Your tent will arrive on Friday, guaranteed."
    - "It'll be with you by 14 March at the latest."
    - "Don't worry, your order arrives tomorrow."
  negatives:
    - "Orders usually arrive in 3 to 5 working days."
    - "Your order shipped on Monday; the tracking link is in your email."
    - "The courier's website shows the latest delivery estimate."
""")
config = {
    "vaccines": {"paths": [str(folder / "vaccines")]},
    "sites": {"order-help": {"enforce": ["pinecrest.no_delivery_promises"]}},
}
cases = [
    ("Your tent will arrive on Friday, guaranteed.", 0.97),
    ("Orders usually arrive in 3 to 5 working days.", 0.03),
]
for reply, jev_says in cases:
    sensor = MockSensor({"promises_date": jev_says})
    with ImmuneHarness(folder / "state", sensor=sensor, script=FakeReply(text=reply), config=config) as harness:
        with immune.site("order-help"):
            answer = harness.openai().chat.completions.create(
                model="gpt-5.4-mini", messages=[{"role": "user", "content": "When will my tent arrive?"}]
            )
        verdict = harness.verdict()
    print(f"model said:    {reply}")
    print(f"customer saw:  {answer.choices[0].message.content}")
    print(f"verdict:       {verdict.action}\n")

model said:    Your tent will arrive on Friday, guaranteed.
customer saw:  Orders usually arrive in 3 to 5 working days; your tracking link is in your email.
verdict:       rewrite

model said:    Orders usually arrive in 3 to 5 working days.
customer saw:  Orders usually arrive in 3 to 5 working days.
verdict:       allow



In your project the same check is an ordinary pytest test. The `immune_harness` fixture is registered when you install
`immune-ai`:

```python
from immune.testing import FakeReply, MockSensor

CONFIG = {
    "vaccines": {"paths": ["vaccines/"]},
    "sites": {"order-help": {"enforce": ["pinecrest.no_delivery_promises"]}},
}


def test_delivery_promises_are_rewritten(immune_harness):
    harness = immune_harness(
        sensor=MockSensor({"promises_date": 0.97}),
        script=FakeReply(text="Your tent will arrive on Friday, guaranteed."),
        config=CONFIG,
    )
    ...
```

To test against Jev's real answers without calling Jev in CI, record them once with `immune.testing.RecordingSensor`
and replay them with `ReplaySensor`; the [testing guide](13_testing_and_ci.ipynb) shows how.

## 8. Ship it observed, then enforce it

A new vaccine starts **observed**: Immune records every hit in the verdict but changes nothing, so you can watch it on
real traffic before it acts. Once the hits look right, enforce it where it matters:

In [7]:
import tempfile
from pathlib import Path

import immune
from immune.testing import FakeReply, ImmuneHarness, MockSensor

folder = Path(tempfile.mkdtemp())
(folder / "vaccines").mkdir()
(folder / "vaccines" / "no_delivery_promises.yaml").write_text("""
id: pinecrest.no_delivery_promises
title: The reply promises a delivery date
description: >-
  Couriers set delivery dates, so the assistant must not promise one. It may quote the
  usual delivery window, share tracking, or say when an order shipped.
stage: output
detect:
  questions:
    - key: promises_date
      text: "The assistant output promises the customer that their order will arrive by a specific date or day."
respond:
  message: "Orders usually arrive in 3 to 5 working days; your tracking link is in your email."
tests:
  positives:
    - "Your tent will arrive on Friday, guaranteed."
    - "It'll be with you by 14 March at the latest."
    - "Don't worry, your order arrives tomorrow."
  negatives:
    - "Orders usually arrive in 3 to 5 working days."
    - "Your order shipped on Monday; the tracking link is in your email."
    - "The courier's website shows the latest delivery estimate."
""")
stages = [
    ("observed", {"vaccines": {"paths": [str(folder / "vaccines")]}}),
    (
        "enforced",
        {
            "vaccines": {"paths": [str(folder / "vaccines")]},
            "sites": {"order-help": {"enforce": ["pinecrest.no_delivery_promises"]}},
        },
    ),
]
for label, config in stages:
    harness = ImmuneHarness(
        folder / label,
        sensor=MockSensor({"promises_date": 0.97}),
        script=FakeReply(text="Your tent will arrive on Friday, guaranteed."),
        config=config,
    )
    with harness, immune.site("order-help"):
        answer = harness.openai().chat.completions.create(
            model="gpt-5.4-mini", messages=[{"role": "user", "content": "When will my tent arrive?"}]
        )
    hit = harness.verdict().hits[0]
    print(f"{label:9} customer saw: {answer.choices[0].message.content}")
    print(f"{'':9} hit: {hit.threat}, probability {hit.probability:.2f}, enforced={hit.enforced}\n")

observed  customer saw: Your tent will arrive on Friday, guaranteed.
          hit: pinecrest.no_delivery_promises, probability 0.97, enforced=False

enforced  customer saw: Orders usually arrive in 3 to 5 working days; your tracking link is in your email.
          hit: pinecrest.no_delivery_promises, probability 0.97, enforced=True



`immune vaccines list --custom` shows each of your vaccines and whether it's on; add `--site` to see one site:

In [8]:
import tempfile
from pathlib import Path

folder = Path(tempfile.mkdtemp())
(folder / "vaccines").mkdir()
(folder / "vaccines" / "no_delivery_promises.yaml").write_text("""
id: pinecrest.no_delivery_promises
title: The reply promises a delivery date
description: >-
  Couriers set delivery dates, so the assistant must not promise one. It may quote the
  usual delivery window, share tracking, or say when an order shipped.
stage: output
detect:
  questions:
    - key: promises_date
      text: "The assistant output promises the customer that their order will arrive by a specific date or day."
respond:
  message: "Orders usually arrive in 3 to 5 working days; your tracking link is in your email."
tests:
  positives:
    - "Your tent will arrive on Friday, guaranteed."
    - "It'll be with you by 14 March at the latest."
    - "Don't worry, your order arrives tomorrow."
  negatives:
    - "Orders usually arrive in 3 to 5 working days."
    - "Your order shipped on Monday; the tracking link is in your email."
    - "The courier's website shows the latest delivery estimate."
""")
(folder / "immune.yaml").write_text("vaccines:\n  paths: [vaccines/]\n")
!cd {folder} && immune vaccines list --custom

protection                      source         stage   detector  floor  state  why
──────────────────────────────  ─────────────  ──────  ────────  ─────  ─────  ─────────────
pinecrest.no_delivery_promises  vaccine 1.0.0  output  jev              on     on by default

1 protections, 1 vaccines loaded, 6 library vaccines available (0 switched on)


## Design checklist

| Before you ship | Why |
| --- | --- |
| The rule is one sentence about one observable behavior | Broad rules fire on everything nearby |
| There are near misses in `tests.negatives` | They're what make it safe to switch on |
| Jev-judged vaccines pass `immune vaccines test --live` | Offline runs check the wiring, not the meaning |
| `immune vaccines trial --corpus your_traffic --live` is quiet | Your traffic is the only real predictor of false alarms |
| Your app's tests cover what users see when it fires | The `message` is part of your product |
| It ships observed first | Real hits are the final review |

Common mistakes:

| Symptom | Likely cause | Fix |
| --- | --- | --- |
| It fires on harmless replies | The question or word list describes the topic, not the behavior | Describe the behavior; add the harmless reply as a near miss |
| A keyword negative "fails" offline | `confirm: jev` needs Jev | Run `immune vaccines test --live` |
| It never fires in production | Wrong `stage`, or `applies_to.sites` doesn't match | `immune vaccines list --custom --site <name>` shows its state |
| A `data` or `tool` question is rejected | Questions there run once per item or call | Refer to the item as `{item}` or the call as `{call}` |

## Recap

- Name one observable rule with an obvious near miss, then pick the place to look and the simplest detector.
- Write positives and near misses: they are the vaccine's specification.
- `immune vaccines test` checks the examples (`--live` for anything Jev judges), and `immune vaccines trial` checks
  everyday traffic.
- Test what your app does when it fires, with the `immune_harness` fixture.
- Ship it observed, read the real hits, then enforce it.

Next: [Testing and CI](13_testing_and_ci.ipynb)